In [4]:
#Verificamos que estamos en el entorno creado para la descarga de datos
import sys
print(sys.executable)


/opt/anaconda3/envs/seguro_cafe/bin/python


In [6]:
#Inicializamos GEE
import importlib
import ee

ee.Reset()
# Inicializar el proyecto correcto y verificamos que está conectado
ee.Initialize(project='redcafe')
print(ee.String('Conexión exitosa con proyecto: redcafe').getInfo())

/opt/anaconda3/envs/seguro_cafe/lib/python3.10/site-packages/google/api_core/_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


Conexión exitosa con proyecto: redcafe


In [8]:
#Probamos la conexión
imagen = ee.Image('ECMWF/ERA5_LAND/DAILY_AGGR/20200101')
print(imagen.bandNames().getInfo())

['dewpoint_temperature_2m', 'temperature_2m', 'skin_temperature', 'soil_temperature_level_1', 'soil_temperature_level_2', 'soil_temperature_level_3', 'soil_temperature_level_4', 'lake_bottom_temperature', 'lake_ice_depth', 'lake_ice_temperature', 'lake_mix_layer_depth', 'lake_mix_layer_temperature', 'lake_shape_factor', 'lake_total_layer_temperature', 'snow_albedo', 'snow_cover', 'snow_density', 'snow_depth', 'snow_depth_water_equivalent', 'snowfall_sum', 'snowmelt_sum', 'temperature_of_snow_layer', 'skin_reservoir_content', 'volumetric_soil_water_layer_1', 'volumetric_soil_water_layer_2', 'volumetric_soil_water_layer_3', 'volumetric_soil_water_layer_4', 'forecast_albedo', 'surface_latent_heat_flux_sum', 'surface_net_solar_radiation_sum', 'surface_net_thermal_radiation_sum', 'surface_sensible_heat_flux_sum', 'surface_solar_radiation_downwards_sum', 'surface_thermal_radiation_downwards_sum', 'evaporation_from_bare_soil_sum', 'evaporation_from_open_water_surfaces_excluding_oceans_sum', '

In [10]:
#Importamos las librerías y definimos las regiones
import ee
import pandas as pd

# Región de Caldas
caldas = ee.Geometry.Rectangle([-76.2, 4.7, -74.9, 5.7])

# Variables que necesitamos de ERA5-Land
variables = [
    'skin_temperature',
    'total_precipitation_sum',
    'volumetric_soil_water_layer_1',
    'temperature_2m_min',
    'temperature_2m_max',
    'dewpoint_temperature_2m',
    'surface_solar_radiation_downwards_sum',  # srad
    'u_component_of_wind_10m',
    'v_component_of_wind_10m',
    'surface_pressure',
]

print("Variables y región definidas")

Variables y región definidas


In [12]:
# Generar lista de fechas de inicio cada 15 días desde 1950 hasta hoy
fechas = []
fecha = pd.Timestamp('2003-01-01')
fin   = pd.Timestamp('2026-04-17')

while fecha < fin:
    fechas.append(fecha.strftime('%Y-%m-%d'))
    fecha += pd.Timedelta(days=16)

print(f"Total períodos de 16 días: {len(fechas)}")
print(f"   Primer período: {fechas[0]}")
print(f"   Último período: {fechas[-1]}")
print(f" Total períodos: {len(fechas)}")

Total períodos de 16 días: 532
   Primer período: 2003-01-01
   Último período: 2026-04-06
 Total períodos: 532


In [14]:
#Definimos una función para crear imagen cada 16 días
def imagen_16dias(fecha_str):
    start = ee.Date(fecha_str)
    end   = start.advance(16, 'day')
    
    col = (ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR")
           .filterDate(start, end)
           .filterBounds(caldas)
           .select(variables))
    
    #Variables que se PROMEDIAN
    img_mean = col.mean()
    
    #Variables que se SUMAN (acumulados)
    img_sum = (ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR")
               .filterDate(start, end)
               .filterBounds(caldas)
               .select([
                   'total_precipitation_sum',
                   'surface_solar_radiation_downwards_sum'
               ])
               .sum()
               .rename(['ppt', 'srad']))
    
    #Temperaturas: Kelvin → Celsius
    tmax  = img_mean.select('temperature_2m_max').subtract(273.15).rename('tmax')
    tmin  = img_mean.select('temperature_2m_min').subtract(273.15).rename('tmin')
    ts_c  = img_mean.select('skin_temperature').subtract(273.15).rename('skin_temp')
    tdew  = img_mean.select('dewpoint_temperature_2m').subtract(273.15).rename('tdew')
    
    #Velocidad del viento (m/s) desde componentes U y V
    u  = img_mean.select('u_component_of_wind_10m')
    v  = img_mean.select('v_component_of_wind_10m')
    ws = u.pow(2).add(v.pow(2)).sqrt().rename('ws')
    
    #Presión superficial
    sp = img_mean.select('surface_pressure').rename('surface_pressure')
    
    # Humedad del suelo
    soil = img_mean.select('volumetric_soil_water_layer_1').rename('soil_water')
    
    #Presión de vapor (vap) desde punto de rocío
    # fórmula Magnus: vap = 0.6108 * exp(17.27 * Td / (Td + 237.3))
    vap = (tdew.multiply(17.27)
               .divide(tdew.add(237.3))
               .exp()
               .multiply(0.6108)
               .rename('vap'))
    
    # VPD = vap_saturación - vap
    tmean    = tmax.add(tmin).divide(2)
    vap_sat  = (tmean.multiply(17.27)
                     .divide(tmean.add(237.3))
                     .exp()
                     .multiply(0.6108))
    vpd = vap_sat.subtract(vap).rename('vpd')
    
    #Unir todas las bandas
    imagen_final = (tmax
                    .addBands(tmin)
                    .addBands(ts_c)
                    .addBands(img_sum.select('ppt'))
                    .addBands(img_sum.select('srad'))
                    .addBands(soil)
                    .addBands(vap)
                    .addBands(vpd)
                    .addBands(ws)
                    .addBands(sp)
                    .set('fecha', fecha_str))
    
    return imagen_final.toFloat()

print("Función definida correctamente")

Función definida correctamente


In [16]:
import time

LOTE = 500

for i, fecha_str in enumerate(fechas): 
    img = imagen_16dias(fecha_str)
    
    task = ee.batch.Export.image.toDrive(
        image          = img,
        description    = f'ERA5_Caldas_{fecha_str}',
        folder         = 'ERA5_Caldas',
        fileNamePrefix = f'ERA5_Caldas_{fecha_str}',
        region         = caldas,
        scale          = 11132,
        crs            = 'EPSG:4326',
        maxPixels      = 1e9
    )
    task.start()
    
    if (i + 1) % LOTE == 0:
        print(f"⏸ Lote {(i+1)//LOTE} completado ({i+1} tareas). Esperando 60s...")
        time.sleep(60)
    
    if (i + 1) % 100 == 0:
        print(f" {i+1} / {len(fechas)} tareas lanzadas...")

print(f"\nTodas las tareas lanzadas")
print("Monitorea en: https://code.earthengine.google.com/tasks")

 100 / 532 tareas lanzadas...
 200 / 532 tareas lanzadas...
 300 / 532 tareas lanzadas...
 400 / 532 tareas lanzadas...
⏸ Lote 1 completado (500 tareas). Esperando 60s...
 500 / 532 tareas lanzadas...

Todas las tareas lanzadas
Monitorea en: https://code.earthengine.google.com/tasks
